# 99 — Durable event release gate

Reconciles archive → Bronze → every populated Silver table → Gold. Missing tables/data, parser failures, coverage gaps, natural-key duplicates/conflicts, timestamp mismatches, cross-epoch discrepancies, SC/identity fields and incorrect factual metrics fail loudly. Replays the common archive and compares every factual card with the backend consumer API, including Start/Final semantics. This is code for Workbench execution, not a claim of prior Workbench success.

Channel coverage defaults are explicit in notebook 10; change them deliberately for recordings with a different approved subscription set. All metrics are checked against reconstruction, not merely table existence.


In [ ]:
# Resolve shared code from the existing notebook in this kernel.
import os
import sys
import json
from pathlib import Path
roots = [Path(os.environ["AIDP_PROJECT_ROOT"])] if os.getenv("AIDP_PROJECT_ROOT") else [Path.cwd(), *Path.cwd().parents]
root = next((p for p in roots if (p / "notebooks_aidp/10_event_capture_foundation.ipynb").is_file()), None)
if root is None:
    raise RuntimeError("Set AIDP_PROJECT_ROOT to the project checkout in Workbench.")
os.environ["AIDP_PROJECT_ROOT"] = str(root.resolve())
foundation = json.loads((root / "notebooks_aidp/10_event_capture_foundation.ipynb").read_text())
for cell in foundation["cells"]:
    if "aidp_shared" in cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(cell["source"]), "10_event_capture_foundation", "exec"), globals())
from pyspark.sql import SparkSession, functions as F, Window
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")


In [ ]:
report = validate_all(spark)
print(report)
